In [1]:
import numpy as np
import pandas as pd
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)

# Metrics
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    average_precision_score,  # PR-AUC
    roc_auc_score
)

# Preprocessing + Model
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Load data - Using the unified master_df.csv
master_path = "../data/processed/master_df.csv"
if not os.path.exists(master_path):
    raise FileNotFoundError(f"Could not find {master_path}. Please run preprocessing_and_features.ipynb first.")

master_df = pd.read_csv(master_path)
master_df['DATE'] = pd.to_datetime(master_df['DATE'])

print("Master dataset shape:", master_df.shape)

label_col = "crash_tomorrow"

# temporal split: <=2023 for training, 2024+ for validation
train_mask = master_df['DATE'].dt.year <= 2023
test_mask  = master_df['DATE'].dt.year > 2023

train_df = master_df[train_mask].copy()
val_df = master_df[test_mask].copy()

print("Train_df shape:", train_df.shape)
print("Val_df shape:  ", val_df.shape)

drop_cols = [
    "geometry",
    "CountyNameTxt",
    "Region",
    "RdwyTypeCde_txtCARTO",
    "DATE"
]

categorical_cols = [
    "WeatherCde_txtCARTO",
    "SurfaceConditionCde_txtCARTO",
]

# Feature Engineering: Extract cell coordinates
for df_tmp in [train_df, val_df]:
    df_tmp[["cell_x", "cell_y"]] = df_tmp["cell_id"].str.split("_", expand=True).astype(int)

# Target and Features
y_train = train_df[label_col].astype(int).values
y_val = val_df[label_col].astype(int).values

X_train_df = train_df.drop(columns=[label_col] + drop_cols + ["cell_id"], errors="ignore")
X_val_df = val_df.drop(columns=[label_col] + drop_cols + ["cell_id"], errors="ignore")

numeric_cols = [c for c in X_train_df.columns if c not in categorical_cols]

print(f"X_train shape: {X_train_df.shape}")
print(f"Numeric features: {len(numeric_cols)}")
print(f"Categorical features: {len(categorical_cols)}")
print("Positive rate (train):", y_train.mean())
print("Positive rate (val):", y_val.mean())


Master dataset shape: (451153, 23)
Train_df shape: (374896, 23)
Val_df shape:   (76257, 23)
X_train shape: (374896, 18)
Numeric features: 16
Categorical features: 2
Positive rate (train): 0.058117451239810505
Positive rate (val): 0.053962259202433874


In [2]:
# Class counts for imbalance handling
n_pos = (y_train == 1).sum()
n_neg = (y_train == 0).sum()

print("Train positives:", n_pos)
print("Train negatives:", n_neg)

# Similar to logistic pos_weight: negatives / positives
pos_weight = n_neg / n_pos
class_weight = {0: 1.0, 1: float(pos_weight)}

print("Computed pos_weight (neg/pos):", pos_weight)
print("Using class_weight:", class_weight)


Train positives: 21788
Train negatives: 353108
Computed pos_weight (neg/pos): 16.206535707729024
Using class_weight: {0: 1.0, 1: 16.206535707729024}


In [ ]:
from sklearn.utils._testing import ignore_warnings
from sklearn.exceptions import ConvergenceWarning
from imblearn.over_sampling import SMOTE

# Define the preprocessor to handle numeric scaling and categorical encoding
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ])

# Preprocess training data
print("Preprocessing training data...")
X_train_processed = preprocessor.fit_transform(X_train_df)

# Apply SMOTE to balance the training set
print("Applying SMOTE to balance class distribution...")
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_processed, y_train)

print(f"Before SMOTE - Positive: {(y_train == 1).sum()}, Negative: {(y_train == 0).sum()}")
print(f"After SMOTE  - Positive: {(y_train_balanced == 1).sum()}, Negative: {(y_train_balanced == 0).sum()}")

# Train MLPClassifier directly on balanced data
print("Training ANN (MLP) model on balanced data...")
ann_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    learning_rate_init=1e-3,
    max_iter=100,
    alpha=1e-4,
    random_state=42,
    verbose=0
)

@ignore_warnings(category=ConvergenceWarning)
def fit_ann_model(model, X, y):
    return model.fit(X, y)

ann_model = fit_ann_model(ann_model, X_train_balanced, y_train_balanced)
print("ANN model trained on balanced data.")

# Also need to preprocess validation data for predictions
print("Preprocessing validation data...")
X_val_processed = preprocessor.transform(X_val_df)


Training ANN (MLP) model...
ANN model trained.


In [ ]:
from sklearn.metrics import confusion_matrix

def hit_rate_at_k(y_true, y_prob, k):
    # Measures what fraction of the top-k highest-probability cells actually had a crash.
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    idx_sorted = np.argsort(y_prob)[::-1]
    top_idx = idx_sorted[:k]
    hits = y_true[top_idx].sum()
    return hits / float(k)


# Get probabilities on validation set using preprocessed data
y_val_proba = ann_model.predict_proba(X_val_processed)[:, 1]
y_val_pred = (y_val_proba >= 0.5).astype(int)

brier = brier_score_loss(y_val, y_val_proba)
logloss = log_loss(y_val, y_val_proba)
pr_auc = average_precision_score(y_val, y_val_proba)
roc_auc = roc_auc_score(y_val, y_val_proba)

# Calculate Sensitivity and Specificity
cm = confusion_matrix(y_val, y_val_pred)
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

# Choose a k for hit-rate
k = 500
hr_k = hit_rate_at_k(y_val, y_val_proba, k)

print("=== ANN (MLP) Model – Validation Metrics (with SMOTE) ===")
print(f"ROC-AUC:            {roc_auc:.6f}")
print(f"Brier score:        {brier:.6f}")
print(f"Log-loss:           {logloss:.6f}")
print(f"PR-AUC (avg prec):  {pr_auc:.6f}")
print(f"Hit-rate@k (k={k}): {hr_k:.6f}")
print(f"Sensitivity:        {sensitivity:.6f}")
print(f"Specificity:        {specificity:.6f}")


=== ANN (MLP) Model – Validation Metrics ===
ROC-AUC:            0.691469
Brier score:        0.049805
Log-loss:           0.198839
PR-AUC (avg prec):  0.117836
Hit-rate@k (k=500): 0.184000
Sensitivity:        0.001215
Specificity:        0.999806
